In [1]:
"""One resumable phase-4C Kaggle training segment; never reads MuSiQue dev."""

import csv
import hashlib
import inspect
import json
import math
import random
import re
import shutil
import time
from pathlib import Path

import numpy as np
import torch
import transformers
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, EarlyStoppingCallback,
                          Trainer, TrainerCallback, TrainingArguments)


BRANCH = "sro"  # Generate a separate notebook with "sro" for the paired branch.
MODE = "train"  # "profile" or "train"; profile does not save a checkpoint.
TARGET_EPOCH = 1  # Train through this epoch, then stop and save for resume.
PRIOR_ROOT = None  # Mounted previous run output directory, or None for epoch 1.
RUN_ID = "phase4c_sro_epoch1_20261122_01"

BASE = Path("/kaggle/input/notebooks/hungluongduc/fact-grounding-gap-deberta-full-musique/fg_full_data")
SRO = Path("/kaggle/input/datasets/hungluongduc/fact-grounding-sro-phase-3-token-audit-20261121")
OUT = Path("/kaggle/working") / f"phase4c_{BRANCH}"
MODEL = "microsoft/deberta-v3-large"
REVISION = "64a8c8eab3e352a784c658aef62be1662607476f"
SEED = 42
SPLIT_HASH = "7C9435008AD7723060EB65FFBEE59722AABD31FC028EDA1E765248414ACFE6F0"
RAW_HASH = "7832387E08D082E4E2CA8B4161AEFEA2CC56450E003E14E312C3BC5F31C0557D"
SRO_HASH = "A94ECA9FA46D41D2F4DE706E03842A30C438F3792A88F9425CF9594EECF85AA8"


def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest().upper()


def rows(path):
    with path.open(encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames != ["text", "label"]:
            raise ValueError(f"invalid CSV header: {path}")
        data = list(reader)
    if len(data) != 46610:
        raise ValueError(f"wrong train rows: {path}")
    return data


def split_key(text):
    prefix, separator = "Sub-question: ", "\n\nPassages:\n"
    if not text.startswith(prefix) or separator not in text:
        raise ValueError("invalid text contract")
    return re.sub(r"\s+", " ", text[len(prefix):].split(separator, 1)[0]).strip().casefold()


def split_indices(raw, sro):
    indices = {"train": [], "validation": []}
    groups = {}
    for i, (a, b) in enumerate(zip(raw, sro)):
        if a["label"] not in {"0", "1"} or a["label"] != b["label"]:
            raise ValueError(f"label mismatch at {i}")
        sep = "\n\nPassages:\n"
        if a["text"].split(sep, 1)[1] != b["text"].split(sep, 1)[1]:
            raise ValueError(f"passage mismatch at {i}")
        key = split_key(a["text"])
        digest = hashlib.sha256(f"{SEED}\n{key}".encode()).digest()
        fold = "validation" if int.from_bytes(digest[:8], "big") < int(0.1 * 2**64) else "train"
        if key in groups and groups[key] != fold:
            raise ValueError("group leakage")
        groups[key] = fold
        indices[fold].append(i)
    # The frozen indices.json was written on Windows with a CRLF final newline.
    payload = (json.dumps(indices, separators=(",", ":")) + "\r\n").encode()
    if hashlib.sha256(payload).hexdigest().upper() != SPLIT_HASH:
        raise ValueError("frozen validation split hash mismatch")
    if len(indices["train"]) != 42211 or len(indices["validation"]) != 4399:
        raise ValueError("frozen validation counts mismatch")
    return indices


class Encoded(torch.utils.data.Dataset):
    def __init__(self, records, tokenizer):
        texts = [r["text"] for r in records]
        self.labels = [int(r["label"]) for r in records]
        self.encodings = tokenizer(texts, max_length=512, truncation=True, padding=False)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {**{k: v[i] for k, v in self.encodings.items()}, "labels": self.labels[i]}


def metrics(prediction):
    probs = torch.softmax(torch.tensor(prediction.predictions), dim=-1)[:, 1].numpy()
    y = np.asarray(prediction.label_ids)
    p = (probs >= 0.5).astype(int)
    tp = int(((y == 1) & (p == 1)).sum())
    fp = int(((y == 0) & (p == 1)).sum())
    fn = int(((y == 1) & (p == 0)).sum())
    tn = int(((y == 0) & (p == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {"precision": precision, "recall": recall,
            "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
            "accuracy": (tp + tn) / len(y), "flag_rate": (tn + fn) / len(y)}


class SegmentStop(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        if state.epoch >= TARGET_EPOCH:
            control.should_training_stop = True
        return control


def main():
    if BRANCH not in {"raw", "sro"} or MODE not in {"profile", "train"}:
        raise ValueError("invalid branch or mode")
    if MODE == "train" and not 1 <= TARGET_EPOCH <= 10:
        raise ValueError("target epoch out of range")
    raw_path, sro_path = BASE / "deberta_fg_train.csv", SRO / "train.csv"
    if sha(raw_path) != RAW_HASH or sha(sro_path) != SRO_HASH:
        raise ValueError("source train CSV hash mismatch")
    raw, sro = rows(raw_path), rows(sro_path)
    indices = split_indices(raw, sro)
    branch_rows = raw if BRANCH == "raw" else sro
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    if torch.cuda.device_count() != 2:
        raise RuntimeError("expected Kaggle T4 x2")
    tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
    train_records = [branch_rows[i] for i in indices["train"]]
    validation_records = [branch_rows[i] for i in indices["validation"]]
    train_ds = Encoded(train_records[:800] if MODE == "profile" else train_records, tokenizer)
    val_ds = Encoded(validation_records, tokenizer) if MODE == "train" else None
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, revision=REVISION, num_labels=2,
        dtype=torch.float32,
        id2label={0: "NOT-ANSWERABLE", 1: "ANSWERABLE"},
        label2id={"NOT-ANSWERABLE": 0, "ANSWERABLE": 1})
    trainable_dtypes = {p.dtype for p in model.parameters() if p.requires_grad}
    if trainable_dtypes != {torch.float32}:
        raise RuntimeError(f"expected FP32 trainable weights for FP16 AMP: {trainable_dtypes}")
    if OUT.exists():
        raise FileExistsError(OUT)
    prior_progress = None
    if PRIOR_ROOT is None:
        OUT.mkdir(parents=True)
        resume = None
    else:
        prior = Path(PRIOR_ROOT)
        prior_progress = json.loads((prior / "progress.json").read_text())
        if prior_progress["branch"] != BRANCH or prior_progress["split_sha256"] != SPLIT_HASH:
            raise ValueError("prior run branch or split mismatch")
        shutil.copytree(prior / "checkpoints", OUT / "checkpoints")
        resume = max((OUT / "checkpoints").glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
    kwargs = dict(output_dir=str(OUT / "checkpoints"), num_train_epochs=10,
                  per_device_train_batch_size=4, per_device_eval_batch_size=4,
                  gradient_accumulation_steps=2, learning_rate=2e-5, weight_decay=0.01,
                  lr_scheduler_type="linear", fp16=True,
                  gradient_checkpointing=True,
                  gradient_checkpointing_kwargs={"use_reentrant": False},
                  optim="adamw_torch", logging_steps=10,
                  disable_tqdm=True, report_to="none", seed=SEED, data_seed=SEED,
                  dataloader_num_workers=2, save_strategy="epoch" if MODE == "train" else "no",
                  load_best_model_at_end=MODE == "train", metric_for_best_model="f1" if MODE == "train" else None,
                  greater_is_better=True, save_total_limit=2,
                  max_steps=50 if MODE == "profile" else -1)
    supported = inspect.signature(TrainingArguments).parameters
    # Transformers 5.20 moved ratio warmup into warmup_steps and renamed
    # group_by_length to train_sampling_strategy. Both express the same setup.
    if "warmup_ratio" in supported:
        kwargs["warmup_ratio"] = 0.1
    else:
        kwargs["warmup_steps"] = 0.1
    if "group_by_length" in supported:
        kwargs["group_by_length"] = True
    else:
        kwargs["train_sampling_strategy"] = "group_by_length"
    strategy_name = "eval_strategy" if "eval_strategy" in supported else "evaluation_strategy"
    kwargs[strategy_name] = "epoch" if MODE == "train" else "no"
    unsupported = set(kwargs) - set(supported)
    if unsupported:
        raise TypeError(f"unsupported TrainingArguments fields: {sorted(unsupported)}")
    args = TrainingArguments(**kwargs)
    callbacks = [SegmentStop()]
    early_stopping = None
    if MODE == "train":
        early_stopping = EarlyStoppingCallback(early_stopping_patience=3)
        if prior_progress:
            early_stopping.early_stopping_patience_counter = prior_progress["early_stopping_patience_counter"]
        callbacks.append(early_stopping)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      compute_metrics=metrics if MODE == "train" else None,
                      data_collator=DataCollatorWithPadding(tokenizer, padding="longest", pad_to_multiple_of=8),
                      callbacks=callbacks)
    started = time.monotonic()
    result = trainer.train(resume_from_checkpoint=str(resume) if resume else None)
    elapsed = time.monotonic() - started
    if MODE == "profile":
        projected = elapsed / trainer.state.global_step * math.ceil(42211 / 16)
        report = {"status": "profile_only", "branch": BRANCH, "steps": trainer.state.global_step,
                  "elapsed_seconds": elapsed, "projected_one_epoch_training_seconds": projected,
                  "under_9_hour_guard": projected < 9 * 3600,
                  "split_sha256": SPLIT_HASH, "model_revision": REVISION,
                  "torch": torch.__version__, "transformers": transformers.__version__}
        (OUT / "profile.json").write_text(json.dumps(report, indent=2) + "\n")
    else:
        state = trainer.state
        report = {"status": "checkpoint_saved", "run_id": RUN_ID, "branch": BRANCH,
                  "target_epoch": TARGET_EPOCH, "completed_epoch": state.epoch,
                  "global_step": state.global_step, "best_validation_f1": state.best_metric,
                  "best_checkpoint": state.best_model_checkpoint,
                  "split_sha256": SPLIT_HASH, "model_revision": REVISION,
                  "raw_sha256": RAW_HASH, "sro_sha256": SRO_HASH,
                  "train_rows": 42211, "validation_rows": 4399,
                  "early_stopping_patience_counter": early_stopping.early_stopping_patience_counter,
                  "training_seconds": elapsed, "train_metrics": result.metrics,
                  "history": state.log_history,
                  "torch": torch.__version__, "transformers": transformers.__version__}
        (OUT / "progress.json").write_text(json.dumps(report, indent=2, default=str) + "\n")
    print(json.dumps(report, indent=2, default=str), flush=True)


if __name__ == "__main__":
    main()


config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  874MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (3 outputs left out)

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.de

{'loss': '2.832', 'grad_norm': '5.145', 'learning_rate': '6.821e-08', 'epoch': '0.00379'}
{'loss': '2.843', 'grad_norm': '11.34', 'learning_rate': '1.44e-07', 'epoch': '0.00758'}
{'loss': '2.801', 'grad_norm': '13.36', 'learning_rate': '2.198e-07', 'epoch': '0.01137'}
{'loss': '2.773', 'grad_norm': '5.249', 'learning_rate': '2.956e-07', 'epoch': '0.01516'}
{'loss': '2.754', 'grad_norm': '17.71', 'learning_rate': '3.714e-07', 'epoch': '0.01895'}
{'loss': '2.81', 'grad_norm': '16.99', 'learning_rate': '4.471e-07', 'epoch': '0.02274'}
{'loss': '2.816', 'grad_norm': '27.3', 'learning_rate': '5.229e-07', 'epoch': '0.02653'}
{'loss': '2.831', 'grad_norm': '7.223', 'learning_rate': '5.987e-07', 'epoch': '0.03032'}
{'loss': '2.798', 'grad_norm': '6.925', 'learning_rate': '6.745e-07', 'epoch': '0.03411'}
{'loss': '2.758', 'grad_norm': '5.238', 'learning_rate': '7.503e-07', 'epoch': '0.0379'}
{'loss': '2.775', 'grad_norm': '6.634', 'learning_rate': '8.261e-07', 'epoch': '0.04169'}
{'loss': '2.80

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '1.368e+04', 'train_samples_per_second': '3.087', 'train_steps_per_second': '0.193', 'train_loss': '2.043', 'epoch': '1'}
{
  "status": "checkpoint_saved",
  "run_id": "phase4c_sro_epoch1_20261122_01",
  "branch": "sro",
  "target_epoch": 1,
  "completed_epoch": 1.0,
  "global_step": 2639,
  "best_validation_f1": 0.762663923887889,
  "best_checkpoint": "/kaggle/working/phase4c_sro/checkpoints/checkpoint-2639",
  "split_sha256": "7C9435008AD7723060EB65FFBEE59722AABD31FC028EDA1E765248414ACFE6F0",
  "model_revision": "64a8c8eab3e352a784c658aef62be1662607476f",
  "raw_sha256": "7832387E08D082E4E2CA8B4161AEFEA2CC56450E003E14E312C3BC5F31C0557D",
  "sro_sha256": "A94ECA9FA46D41D2F4DE706E03842A30C438F3792A88F9425CF9594EECF85AA8",
  "train_rows": 42211,
  "validation_rows": 4399,
  "early_stopping_patience_counter": 0,
  "training_seconds": 13680.555145803,
  "train_metrics": {
    "train_runtime": 13675.5339,
    "train_samples_per_second": 3.087,
    "train_steps_per_second"